# ALS-based Collaborative Filtering Recommender (Implicit Feedback)

Objective:
- Recommend Top-K movies for each user based on implicit viewing behavior.
- Explore collaborative filtering under extreme data sparsity.

Key Challenges:
- Highly sparse user–item interaction matrix
- Very limited interactions per user
- Strong popularity bias in implicit feedback settings

Approach:
- Construct weighted implicit feedback signals using viewing progress,
  rewatch behavior, and recency.
- Train an ALS (Alternating Least Squares) model using the implicit library.
- Evaluate recommendations with Recall@K and NDCG@K under a temporal split.


# 0. Library & Data Loading

In [ ]:
%pip install implicit

import pandas as pd
import numpy as np
import scipy.sparse as sp
import implicit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.4/761.4 kB 9.1 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


/opt/homebrew/Caskroom/miniforge/base/envs/eda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
his = pd.read_csv("/Users/injoo/Desktop/netflix_project/watch_history_preprocessed.csv")
movies = pd.read_csv("/Users/injoo/Desktop/netflix_project/movies_preprocessed.csv")

# 1. Data Cleaning (Core Steps)

In [ ]:
# Remove watch logs for movies not present in the movies table
valid_movie_ids = set(movies["movie_id"])
his = his[his["movie_id"].isin(valid_movie_ids)]

# Remove movies with invalid runtime (runtime = 0)
bad_movies = movies.loc[movies["runtime_min"] == 0, "movie_id"]
movies = movies[~movies["movie_id"].isin(bad_movies)]
his = his[~his["movie_id"].isin(bad_movies)]


# 2. Implicit Feedback Weight Design
- Core idea:
- Convert viewing behavior into confidence scores for implicit ALS

In [6]:
# Progress-based weight
bins = [0, 20, 40, 60, 80, 100]
labels = ["0~20", "20~40", "40~60", "60~80", "80~100"]
weight_map = {
    "0~20": 1.1,
    "20~40": 1.3,
    "40~60": 1.5,
    "60~80": 1.8,
    "80~100": 2.0
}

his["progress_bin"] = pd.cut(his["progress_percentage"], bins=bins, labels=labels)
his["progress_weight"] = his["progress_bin"].map(weight_map).astype(float).fillna(1.0)


In [7]:
# Rewatch weight
watch_cnt = his.groupby(["user_id","movie_id"]).size().reset_index(name="cnt")
watch_cnt["rewatch_weight"] = watch_cnt["cnt"].apply(
    lambda x: 1.0 if x==1 else 1.5 if x==2 else 2.0
)

his = his.merge(
    watch_cnt[["user_id","movie_id","rewatch_weight"]],
    on=["user_id","movie_id"],
    how="left"
)

In [8]:
# Recency weight
his["watch_date"] = pd.to_datetime(his["watch_date"])
ref = his["watch_date"].max()
his["days_since"] = (ref - his["watch_date"]).dt.days

his["recency_weight"] = 1.0 + np.exp(-0.005 * his["days_since"])

In [9]:
# Final implicit confidence
his["final_weight"] = (
    his["progress_weight"]
    + his["rewatch_weight"]
    + his["recency_weight"]
)

# 3. User–Item Aggregation & Temporal Split

In [ ]:
ui = (
    his.groupby(["user_id","movie_id"], as_index=False)
       .agg({"final_weight":"max", "watch_date":"max"})
)


In [ ]:
# Rank interactions by recency per user
ui["rank_latest"] = ui.groupby("user_id")["watch_date"]\
                      .rank(ascending=False, method="first")


# Temporal split
train = ui[ui["rank_latest"] >= 3].copy()
val   = ui[ui["rank_latest"] == 2].copy()
test  = ui[ui["rank_latest"] == 1].copy()


# 4. Sparse Matrix Construction & ALS Training

In [ ]:
# Map user and item IDs to continuous indices
user2idx = {u:i for i,u in enumerate(train["user_id"].unique())}
movie2idx = {m:i for i,m in enumerate(train["movie_id"].unique())}

for df in [train, val, test]:
    df["user_idx"] = df["user_id"].map(user2idx)
    df["movie_idx"] = df["movie_id"].map(movie2idx)

# Drop cold users/items not present in training
val.dropna(subset=["user_idx", "movie_idx"], inplace=True)
test.dropna(subset=["user_idx", "movie_idx"], inplace=True)

# Convert indices to int
for df in [train, val, test]:
    df["user_idx"] = df["user_idx"].astype(int)
    df["movie_idx"] = df["movie_idx"].astype(int)

In [ ]:
# Build sparse user–item matrix
train_mat = sp.coo_matrix(
    (train["final_weight"], (train["user_idx"], train["movie_idx"]))
).tocsr()

In [ ]:
model = implicit.als.AlternatingLeastSquares(
    factors=100,
    regularization=0.01,
    iterations=100,
    random_state=42
)

# implicit expects item-user matrix
model.fit(train_mat.T)

/opt/homebrew/Caskroom/miniforge/base/envs/eda/lib/python3.10/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
/opt/homebrew/Caskroom/miniforge/base/envs/eda/lib/python3.10/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed csc_matrix instead. Converting to CSR took 0.08403515815734863 seconds
  warnings.warn(
python(64183) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
100%|██████████| 100/100 [00:09<00:00, 10.80it/s]


# 5. Evaluation Metrics

### Evaluation Objective

The recommender system is evaluated based on how well it predicts
the items that users actually consumed.

Given the implicit-feedback setting, the following metrics are used:

- **Recall@K**:  
  Whether the item a user actually watched appears in the Top-K recommendations.

- **NDCG@K**:  
  Whether the correctly predicted item is ranked near the top of the recommendation list.

These metrics are widely used in offline evaluation
for implicit recommendation systems.

### Evaluation Setup

The dataset is split temporally at the user level:

- **Train**: Past viewing history
- **Validation**: Second most recent interaction (for tuning)
- **Test**: Most recent interaction per user

Each user has **at most one test item**,
making the evaluation particularly strict.

### Recall@K

Recall@K is defined as:

Recall@K = (Number of relevant items in Top-K) / (Number of relevant items)

Since each user has only one test item,
Recall@K becomes a binary outcome per user:
- 1 if the test item appears in Top-K
- 0 otherwise

The final score is averaged across users.

In [23]:
from collections import defaultdict
import numpy as np

def recall_at_k(model, train_mat, test_df, k=10):
    """
    Compute Recall@K for implicit ALS model
    """
    # 유저별 실제 Test 아이템
    test_user_items = defaultdict(set)
    for row in test_df.itertuples():
        test_user_items[row.user_idx].add(row.movie_idx)

    recalls = []
    users = test_df["user_idx"].unique()

    for user in users:
        # Top-K 추천
        recommended, _ = model.recommend(
            userid=user,
            user_items=train_mat[user],
            N=k,
            filter_already_liked_items=True
        )

        true_items = test_user_items[user]
        if len(true_items) == 0:
            continue

        hits = len(set(recommended) & true_items)
        recalls.append(hits / len(true_items))

    return np.mean(recalls) if recalls else 0.0


### NDCG@K

NDCG@K (Normalized Discounted Cumulative Gain)
accounts for the ranking position of the correct item.

Correct predictions appearing earlier in the list
receive higher scores through logarithmic discounting.

This metric captures not only correctness
but also ranking quality.



In [25]:
def ndcg_at_k(model, train_mat, test_df, k=10):
    from collections import defaultdict
    import numpy as np

    test_user_items = defaultdict(set)
    for row in test_df.itertuples():
        test_user_items[row.user_idx].add(row.movie_idx)

    ndcgs = []
    users = test_df["user_idx"].unique()

    for user in users:
        recommended, _ = model.recommend(
            userid=user,
            user_items=train_mat[user],
            N=k,
            filter_already_liked_items=True
        )

        true_items = test_user_items[user]
        if not true_items:
            continue

        dcg = 0.0
        for i, item in enumerate(recommended):
            if item in true_items:
                dcg += 1.0 / np.log2(i + 2)

        idcg = sum(
            1.0 / np.log2(i + 2)
            for i in range(min(len(true_items), k))
        )

        if idcg > 0:
            ndcgs.append(dcg / idcg)

    return np.mean(ndcgs) if ndcgs else 0.0


In [27]:
max_user_idx = model.user_factors.shape[0]

test_eval = test[test["user_idx"] < max_user_idx].copy()


In [29]:
k = 10
recall = recall_at_k(model, train_mat, test_eval, k)
ndcg = ndcg_at_k(model, train_mat, test_eval, k)

print(f"Recall@{k}: {recall:.4f}")
print(f"NDCG@{k}: {ndcg:.4f}")

Recall@10: 0.0010
NDCG@10: 0.0003


# 6. Performance Interpretation

At first glance, Recall@10 and NDCG@10 appear extremely low.
However, this behavior is expected given the structure of the dataset.

Key limiting factors include:

- Very few interactions per user (≈ 7–8 on average)
- Only one test item per user
- Extremely sparse user–item matrix (≈ 0.8% observed entries)
- Strong popularity bias under limited interaction data

In such environments, offline metrics based on Recall@K
tend to produce near-zero values even for reasonable models.

Therefore, these results reflect the **limitations of the data and evaluation setup**
rather than a failure of the modeling approach.